In [3]:
!pip install pyspark prophet plotly duckdb openpyxl -q

**Mount Google Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Upload Files**

In [5]:
from google.colab import files
uploaded = files.upload()

Saving online_retail_II.csv to online_retail_II.csv


**Load CSV Files**

In [6]:
import pandas as pd

df_raw = pd.read_csv('online_retail_II.csv')
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (1067371, 8)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Start PySpark Session**





In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SupplyChainProject") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark version:", spark.version)
print("✅ Spark is running!")

Spark version: 4.0.2
✅ Spark is running!


**Convert to Spark DataFrame** *(bronze layer)*

In [8]:
df_bronze = spark.createDataFrame(df_raw)

print(f"Total rows (Bronze): {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.show(5)

Total rows (Bronze): 1067371
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|RECOR

 **Clean Data** *(silver layer)*

In [9]:
from pyspark.sql.functions import col, round as spark_round, month, year

df_silver = df_bronze \
    .dropna(subset=["Customer ID", "Invoice", "StockCode"]) \
    .filter(col("Quantity") > 0) \
    .filter(col("Price") > 0) \
    .withColumn("Revenue", spark_round(col("Quantity") * col("Price"), 2)) \
    .withColumn("InvoiceDate", col("InvoiceDate").cast("timestamp")) \
    .withColumn("Month", month(col("InvoiceDate"))) \
    .withColumn("Year", year(col("InvoiceDate")))

print(f"Clean rows (Silver): {df_silver.count()}")
df_silver.show(5)

Clean rows (Silver): 805549
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+-----+----+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|Revenue|Month|Year|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+-------+-----+----+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|   83.4|   12|2009|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|   81.0|   12|2009|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|   81.0|   12|2009|
| 489434|    22041|RECORD FRAME 7" S...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|  100.8|   12|2009|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|   30.0|   12|20

**Aggregate Data** *(gold layer)*

In [10]:
from pyspark.sql.functions import countDistinct, sum as spark_sum

df_gold = df_silver.groupBy("Country", "Year", "Month") \
    .agg(
        spark_round(spark_sum("Revenue"), 2).alias("Total_Revenue"),
        spark_sum("Quantity").alias("Total_Units"),
        countDistinct("Invoice").alias("Total_Orders"),
        countDistinct("Customer ID").alias("Unique_Customers")
    ) \
    .orderBy("Year", "Month")

print(f"Gold layer rows: {df_gold.count()}")
df_gold.show(10)

Gold layer rows: 546
+-----------+----+-----+-------------+-----------+------------+----------------+
|    Country|Year|Month|Total_Revenue|Total_Units|Total_Orders|Unique_Customers|
+-----------+----+-----+-------------+-----------+------------+----------------+
|    Belgium|2009|   12|        447.6|        153|           3|               2|
|     Greece|2009|   12|       610.95|        441|           1|               1|
|     Poland|2009|   12|       371.82|        266|           1|               1|
|     France|2009|   12|      6521.69|       3688|          13|              10|
|        USA|2009|   12|        141.0|          1|           1|               1|
|   Portugal|2009|   12|      2821.58|       1575|           2|               2|
|Netherlands|2009|   12|     15204.73|      10822|           6|               2|
|    Austria|2009|   12|      1998.34|        564|           2|               2|
|     Cyprus|2009|   12|      3556.98|       1578|           4|               3|
|      

**Save Gold Layer to Drive as Parquet**

In [11]:
df_gold.write.mode("overwrite").parquet("/content/drive/MyDrive/supply_chain_project/gold_sales")

print("Gold layer saved as Parquet to Google Drive!")

Gold layer saved as Parquet to Google Drive!


**Export CSV for Power BI**

In [12]:
df_gold_pd = df_gold.toPandas()
df_gold_pd.to_csv("/content/drive/MyDrive/supply_chain_project/gold_sales.csv", index=False)

print("CSV exported! Rows:", len(df_gold_pd))
df_gold_pd.head()

CSV exported! Rows: 546


,Country,Year,Month,Total_Revenue,Total_Units,Total_Orders,Unique_Customers
0,United Kingdom,2009,12,613214.90,361783,1429,904
1,France,2009,12,6521.69,3688,13,10
2,Australia,2009,12,271.10,160,2,2
3,Portugal,2009,12,2821.58,1575,2,2
4,Norway,2009,12,485.31,182,1,1


**Time Series Data**

In [13]:
df_ts = df_gold_pd.copy()
df_ts["ds"] = pd.to_datetime(df_ts[["Year", "Month"]].assign(Day=1))
df_ts = df_ts.groupby("ds")["Total_Revenue"].sum().reset_index()
df_ts.columns = ["ds", "y"]
df_ts = df_ts.sort_values("ds").reset_index(drop=True)

print("Time series shape:", df_ts.shape)
print(df_ts)

Time series shape: (25, 2)
           ds           y
0  2009-12-01   686654.16
1  2010-01-01   557319.06
2  2010-02-01   506371.06
3  2010-03-01   699608.99
4  2010-04-01   594609.19
5  2010-05-01   599985.79
6  2010-06-01   639066.58
7  2010-07-01   591636.74
8  2010-08-01   604242.65
9  2010-09-01   831615.00
10 2010-10-01  1036680.00
11 2010-11-01  1172336.04
12 2010-12-01   884591.89
13 2011-01-01   569445.04
14 2011-02-01   447137.35
15 2011-03-01   595500.76
16 2011-04-01   469200.36
17 2011-05-01   678594.56
18 2011-06-01   661213.69
19 2011-07-01   600091.01
20 2011-08-01   645343.90
21 2011-09-01   952838.38
22 2011-10-01  1039318.79
23 2011-11-01  1161817.38
24 2011-12-01   518210.79


**Train Prophet Model + Forecast**

In [14]:
from prophet import Prophet

model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.fit(df_ts)

future = model.make_future_dataframe(periods=6, freq="MS")
forecast = model.predict(future)

print(" Model trained!")
print(f"Forecasting up to: {forecast['ds'].max().strftime('%B %Y')}")
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(8)

INFO:prophet:n_changepoints greater than number of observations. Using 19.


 Model trained!
Forecasting up to: June 2012


,ds,yhat,yhat_lower,yhat_upper
23,2011-11-01,1.159621e+06,1.093568e+06,1.226886e+06
24,2011-12-01,6.191605e+05,5.477546e+05,6.893124e+05
25,2012-01-01,6.599635e+05,5.961987e+05,7.243985e+05
26,2012-02-01,5.035481e+05,4.339048e+05,5.693439e+05
27,2012-03-01,6.853320e+05,6.177132e+05,7.519321e+05
28,2012-04-01,7.161403e+05,6.516107e+05,7.823401e+05
29,2012-05-01,4.064074e+05,3.406758e+05,4.738074e+05
30,2012-06-01,6.391357e+05,5.746134e+05,7.064536e+05


**Plot Forecast Chart**

In [15]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_scatter(
    x=df_ts["ds"], y=df_ts["y"],
    name="Actual Revenue",
    line=dict(color="#f5c842", width=2)
)
fig.add_scatter(
    x=forecast["ds"], y=forecast["yhat"],
    name="Forecast",
    line=dict(color="#5ce0b8", width=2, dash="dash")
)
fig.add_scatter(
    x=forecast["ds"], y=forecast["yhat_upper"],
    fill=None, line=dict(color="rgba(92,224,184,0.1)"),
    name="Upper Bound", showlegend=False
)
fig.add_scatter(
    x=forecast["ds"], y=forecast["yhat_lower"],
    fill="tonexty", line=dict(color="rgba(92,224,184,0.1)"),
    name="Confidence Band"
)

fig.update_layout(
    title="📦 Supply Chain Demand Forecast — Next 6 Months",
    xaxis_title="Date",
    yaxis_title="Revenue (£)",
    template="plotly_dark",
    height=500
)

fig.show()
fig.write_html("/content/drive/MyDrive/supply_chain_project/forecast_chart.html")
print(" Chart saved to Drive!")

 Chart saved to Drive!


**Model Accuracy**

In [16]:
from sklearn.metrics import mean_absolute_percentage_error

actual = df_ts["y"].values
predicted = forecast["yhat"][:len(actual)].values

mape = mean_absolute_percentage_error(actual, predicted)
print(f" Model MAPE: {mape:.2%}")
print(f" Model Accuracy: {(1 - mape) * 100:.1f}%")
print()
print(f"   'Demand forecasting model achieving {mape:.1%} MAPE'")

 Model MAPE: 5.37%
 Model Accuracy: 94.6%

   'Demand forecasting model achieving 5.4% MAPE'


**Top 10 Countries by Revenue**

In [17]:
top_countries = df_gold_pd.groupby("Country")["Total_Revenue"] \
    .sum().reset_index() \
    .sort_values("Total_Revenue", ascending=False) \
    .head(10)

fig2 = go.Figure(go.Bar(
    x=top_countries["Total_Revenue"],
    y=top_countries["Country"],
    orientation="h",
    marker_color="#f5c842"
))

fig2.update_layout(
    title="Top 10 Countries by Revenue",
    template="plotly_dark",
    height=400
)

fig2.show()

**Install DuckDB**

In [18]:
import duckdb
conn = duckdb.connect()
conn.execute("CREATE TABLE gold_sales AS SELECT * FROM df_gold_pd")
print(" SQL database ready!")

 SQL database ready!


**Total Revenue by Country**

In [19]:
result1 = conn.execute("""
    SELECT
        Country,
        SUM(Total_Revenue) AS Total_Revenue,
        SUM(Total_Orders) AS Total_Orders,
        SUM(Unique_Customers) AS Unique_Customers,
        ROUND(SUM(Total_Revenue) / SUM(Total_Orders), 2) AS Avg_Order_Value
    FROM gold_sales
    GROUP BY Country
    ORDER BY Total_Revenue DESC
    LIMIT 10
""").df()

print(" Top 10 Countries by Revenue:")
result1

 Top 10 Countries by Revenue:


,Country,Total_Revenue,Total_Orders,Unique_Customers,Avg_Order_Value
0,United Kingdom,14723147.50,33541.0,23417.0,438.96
1,EIRE,621631.11,567.0,62.0,1096.35
2,Netherlands,554232.34,228.0,94.0,2430.84
3,Germany,431262.46,789.0,560.0,546.59
4,France,355257.47,614.0,468.0,578.60
5,Australia,169968.11,95.0,71.0,1789.14
6,Spain,109178.53,154.0,128.0,708.95
7,Switzerland,100365.34,90.0,75.0,1115.17
8,Sweden,91549.72,104.0,62.0,880.29
9,Denmark,69862.19,43.0,40.0,1624.70


**Month-over-Month Growth**

In [20]:
result2 = conn.execute("""
    SELECT
        Year,
        Month,
        SUM(Total_Revenue) AS Monthly_Revenue,
        LAG(SUM(Total_Revenue)) OVER (ORDER BY Year, Month) AS Prev_Month_Revenue,
        ROUND(
            (SUM(Total_Revenue) - LAG(SUM(Total_Revenue)) OVER (ORDER BY Year, Month))
            / LAG(SUM(Total_Revenue)) OVER (ORDER BY Year, Month) * 100
        , 2) AS MoM_Growth_Pct
    FROM gold_sales
    GROUP BY Year, Month
    ORDER BY Year, Month
""").df()

print(" Month-over-Month Revenue Growth:")
result2

 Month-over-Month Revenue Growth:


,Year,Month,Monthly_Revenue,Prev_Month_Revenue,MoM_Growth_Pct
0,2009,12,686654.16,NaN,NaN
1,2010,1,557319.06,686654.16,-18.84
2,2010,2,506371.06,557319.06,-9.14
3,2010,3,699608.99,506371.06,38.16
4,2010,4,594609.19,699608.99,-15.01
5,2010,5,599985.79,594609.19,0.90
6,2010,6,639066.58,599985.79,6.51
7,2010,7,591636.74,639066.58,-7.42
8,2010,8,604242.65,591636.74,2.13
9,2010,9,831615.00,604242.65,37.63


**Best & Worst Performing Months**

In [21]:
result3 = conn.execute("""
    SELECT
        CASE Month
            WHEN 1 THEN 'January' WHEN 2 THEN 'February'
            WHEN 3 THEN 'March' WHEN 4 THEN 'April'
            WHEN 5 THEN 'May' WHEN 6 THEN 'June'
            WHEN 7 THEN 'July' WHEN 8 THEN 'August'
            WHEN 9 THEN 'September' WHEN 10 THEN 'October'
            WHEN 11 THEN 'November' WHEN 12 THEN 'December'
        END AS Month_Name,
        ROUND(AVG(Total_Revenue), 2) AS Avg_Revenue,
        ROUND(AVG(Total_Orders), 2) AS Avg_Orders
    FROM gold_sales
    GROUP BY Month
    ORDER BY Avg_Revenue DESC
""").df()

print("Revenue by Month (avg across all years):")
result3

Revenue by Month (avg across all years):


,Month_Name,Avg_Revenue,Avg_Orders
0,November,47635.78,107.02
1,October,42367.32,82.90
2,September,39654.52,76.53
3,December,33700.92,59.52
4,May,32784.11,75.18
5,March,30118.83,66.16
6,January,28891.39,51.23
7,August,28399.69,58.48
8,June,28266.96,62.83
9,July,27084.72,61.64


 **Customer Retention Insight**

In [22]:
result4 = conn.execute("""
    SELECT
        Year,
        SUM(Unique_Customers) AS Total_Customers,
        SUM(Total_Orders) AS Total_Orders,
        ROUND(SUM(Total_Orders) * 1.0 / SUM(Unique_Customers), 2) AS Orders_Per_Customer,
        ROUND(SUM(Total_Revenue) / SUM(Unique_Customers), 2) AS Revenue_Per_Customer
    FROM gold_sales
    GROUP BY Year
    ORDER BY Year
""").df()

print(" Customer Behaviour by Year:")
result4

 Customer Behaviour by Year:


,Year,Total_Customers,Total_Orders,Orders_Per_Customer,Revenue_Per_Customer
0,2009,955.0,1512.0,1.58,719.01
1,2010,12476.0,18325.0,1.47,698.79
2,2011,12172.0,17132.0,1.41,685.07


**Export All SQL Results to Excel**

In [23]:
output_path = "/content/drive/MyDrive/supply_chain_project/sql_analysis.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    result1.to_excel(writer, sheet_name='Revenue by Country', index=False)
    result2.to_excel(writer, sheet_name='MoM Growth', index=False)
    result3.to_excel(writer, sheet_name='Best Months', index=False)
    result4.to_excel(writer, sheet_name='Customer Insights', index=False)
    df_gold_pd.to_excel(writer, sheet_name='Full Gold Data', index=False)

print(" Excel file saved with 5 sheets to Google Drive!")
print(" File: sql_analysis.xlsx")

 Excel file saved with 5 sheets to Google Drive!
 File: sql_analysis.xlsx


**Forecast Data**

In [25]:
# Export forecast data for Power BI
forecast_export = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
forecast_export.columns = ["Date", "Forecasted_Revenue", "Lower_Bound", "Upper_Bound"]

actual_export = df_ts.copy()
actual_export.columns = ["Date", "Actual_Revenue"]

final_forecast = forecast_export.merge(actual_export, on="Date", how="left")
final_forecast.to_csv("/content/drive/MyDrive/supply_chain_project/forecast_data.csv", index=False)

from google.colab import files
files.download("forecast_data.csv")
print(" Done! Check your Downloads folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Done! Check your Downloads folder
